# 02 — Canonical time-series EDA

**Purpose:** understand the calibration data before modelling. The notebook
uses `SPEC-CORE` only. It reports structure, data quality, representative
distributions and series, temporal dependence, possible seasonality,
baseline-review flags and detector readiness.

It does not use labels, choose a model or tune an alert threshold.


## 1. Setup and sector switch


In [ ]:
import importlib
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")
    or ("/content/drive/MyDrive/anomaly_detection" if IN_COLAB
        else Path.home() / "anomaly_detection_data")
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")  # or "petrobras_3w"
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_10_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_10_1_run2",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json

CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
EVAL_ROOT = RUN_ROOT / "SPEC-EVAL"
SPLIT_ROOT = RUN_ROOT / "SPLITS"

import duckdb
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import STL

EDA_VERSION = "2.1.0"
EDA_RUN_ID = os.getenv("EDA_RUN_ID", f"{SECTOR}_eda_v2_1_run1")
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{EDA_VERSION}" / SECTOR / EDA_RUN_ID
MAX_SAMPLE_ROWS = int(os.getenv("EDA_MAX_SAMPLE_ROWS", "300000"))
MAX_PLOT_METRICS = int(os.getenv("EDA_MAX_PLOT_METRICS", "6"))

if not CORE_ROOT.is_dir():
    raise FileNotFoundError(f"Run 01B first: {CORE_ROOT}")

manifest = read_json(CORE_ROOT / "manifest.json")
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
registry = pd.read_parquet(CORE_ROOT / "entity_registry.parquet")
telemetry_glob = str(CORE_ROOT / "telemetry" / "part-*.parquet")

time_path = SPLIT_ROOT / "time_partitions.parquet"
entity_path = SPLIT_ROOT / "entity_partitions.parquet"
primary_split = "time" if time_path.is_file() else "entity"
connection = duckdb.connect()
telemetry_sql = telemetry_glob.replace("'", "''")
connection.execute(f"CREATE VIEW telemetry AS SELECT * FROM read_parquet('{telemetry_sql}')")
if primary_split == "time":
    partitions = pd.read_parquet(time_path)
    calibration = partitions.loc[partitions["partition"].eq("calibration")].iloc[0]
    where = "event_ts >= ? AND event_ts < ?"
    qualified_where = "t.event_ts >= ? AND t.event_ts < ?"
    parameters = [pd.to_datetime(calibration.start_ts, utc=True), pd.to_datetime(calibration.end_ts, utc=True)]
else:
    partitions = pd.read_parquet(entity_path)
    calibration_entities = partitions.loc[partitions["partition"].eq("calibration"), ["entity_id"]]
    connection.register("calibration_entities", calibration_entities)
    where = "CAST(entity_id AS VARCHAR) IN (SELECT CAST(entity_id AS VARCHAR) FROM calibration_entities)"
    qualified_where = "CAST(t.entity_id AS VARCHAR) IN (SELECT CAST(entity_id AS VARCHAR) FROM calibration_entities)"
    parameters = []

display(pd.Series({
    "sector": SECTOR,
    "analysis_partition": "calibration",
    "split": primary_split,
    "canonical_input": str(CORE_ROOT),
    "eda_output": str(EDA_ROOT),
}, name="value").to_frame())


## 2. Structure and data quality


In [ ]:
metric_summary = connection.execute(f"""
    SELECT metric_id,
           count(*) AS rows,
           count(DISTINCT entity_id) AS entities,
           count(DISTINCT episode_id) AS episodes,
           avg((quality_code = 'invalid' OR value IS NULL)::INTEGER) AS invalid_rate,
           avg((quality_code = 'clipped')::INTEGER) AS clipped_rate,
           approx_quantile(value, 0.01) FILTER (WHERE quality_code <> 'invalid') AS q01,
           approx_quantile(value, 0.50) FILTER (WHERE quality_code <> 'invalid') AS median,
           approx_quantile(value, 0.99) FILTER (WHERE quality_code <> 'invalid') AS q99
    FROM telemetry
    WHERE {where}
    GROUP BY metric_id
    ORDER BY metric_id
""", parameters).df().merge(catalogue, on="metric_id", how="left", validate="one_to_one")

series_summary = connection.execute(f"""
    SELECT CAST(entity_id AS VARCHAR) AS entity_id,
           CAST(episode_id AS VARCHAR) AS episode_id,
           metric_id,
           count(*) AS rows,
           avg((quality_code <> 'invalid' AND value IS NOT NULL)::INTEGER) AS valid_rate,
           min(event_ts) AS observed_from,
           max(event_ts) AS observed_to
    FROM telemetry
    WHERE {where}
    GROUP BY entity_id, episode_id, metric_id
""", parameters).df()

display(metric_summary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
metric_summary.plot.bar(x="metric_id", y="invalid_rate", ax=axes[0], legend=False)
metric_summary.plot.bar(x="metric_id", y="clipped_rate", ax=axes[1], legend=False)
axes[0].set_title("Invalid or missing values")
axes[1].set_title("Clipped values")
for axis in axes:
    axis.set_ylabel("fraction")
    axis.tick_params(axis="x", rotation=75)
plt.tight_layout()
plt.show()


## 3. Distributions and representative time series


In [ ]:
sample = connection.execute(f"""
    SELECT * FROM (
        SELECT event_ts, CAST(entity_id AS VARCHAR) AS entity_id,
               CAST(episode_id AS VARCHAR) AS episode_id,
               metric_id, value, quality_code
        FROM telemetry
        WHERE {where}
          AND quality_code <> 'invalid' AND value IS NOT NULL
    ) AS eligible
    USING SAMPLE reservoir({MAX_SAMPLE_ROWS} ROWS) REPEATABLE (42)
""", parameters).df()
sample["event_ts"] = pd.to_datetime(sample["event_ts"], utc=True)

plot_metrics = metric_summary.sort_values("rows", ascending=False)["metric_id"].head(MAX_PLOT_METRICS).tolist()
representatives = (
    series_summary.loc[series_summary["metric_id"].isin(plot_metrics)]
    .sort_values(["metric_id", "valid_rate", "rows"], ascending=[True, False, False])
    .groupby("metric_id", as_index=False)
    .first()[["entity_id", "episode_id", "metric_id"]]
)
connection.register("representatives", representatives)
representative_data = connection.execute(f"""
    SELECT t.event_ts, CAST(t.entity_id AS VARCHAR) AS entity_id,
           CAST(t.episode_id AS VARCHAR) AS episode_id,
           t.metric_id, t.value
    FROM telemetry AS t
    JOIN representatives AS r
      ON CAST(t.entity_id AS VARCHAR) = r.entity_id
     AND CAST(t.episode_id AS VARCHAR) = r.episode_id
     AND t.metric_id = r.metric_id
    WHERE {qualified_where}
      AND t.quality_code <> 'invalid' AND t.value IS NOT NULL
    ORDER BY t.metric_id, t.event_ts
""", parameters).df()
representative_data["event_ts"] = pd.to_datetime(
    representative_data["event_ts"], utc=True
)

fig, axes = plt.subplots(len(plot_metrics), 2, figsize=(13, 3.2 * len(plot_metrics)))
axes = np.atleast_2d(axes)
for row, metric_id in enumerate(plot_metrics):
    values = sample.loc[sample["metric_id"].eq(metric_id), "value"].dropna()
    low, high = values.quantile([0.005, 0.995]) if len(values) else (np.nan, np.nan)
    axes[row, 0].hist(values[values.between(low, high)], bins=40)
    axes[row, 0].set_title(f"{metric_id}: central 99%")

    chosen = representatives.loc[representatives["metric_id"].eq(metric_id)]
    series = representative_data.loc[
        representative_data["metric_id"].eq(metric_id), ["event_ts", "value"]
    ]
    if chosen.empty or series.empty:
        continue
    if len(series) > 5000:
        series = series.iloc[::max(1, len(series) // 5000)]
    axes[row, 1].plot(series.event_ts, series.value, linewidth=0.8)
    axes[row, 1].set_title(f"{metric_id}: {chosen.iloc[0].entity_id}")
plt.tight_layout()
plt.show()


## 4. Temporal dependence and seasonality

ACF and STL are descriptive checks, not automatic feature selection. The
longest regular calibration segment is used; gaps are not interpolated
across. STL is attempted only when at least six complete candidate cycles
are available.


In [ ]:
temporal_rows = []
for metric_id in plot_metrics:
    meta = catalogue.set_index("metric_id").loc[metric_id]
    cadence = pd.to_numeric(meta.expected_cadence_seconds, errors="coerce")
    chosen = representatives.loc[representatives.metric_id.eq(metric_id)]
    frame = representative_data.loc[
        representative_data.metric_id.eq(metric_id), ["event_ts", "value"]
    ].copy()
    if chosen.empty or frame.empty or pd.isna(cadence):
        continue
    chosen = chosen.iloc[0]
    breaks = frame.event_ts.diff().dt.total_seconds().gt(1.5 * cadence).cumsum()
    segment = max((part for _, part in frame.groupby(breaks)), key=len)
    values = segment.set_index("event_ts").value.asfreq(pd.Timedelta(seconds=float(cadence))).dropna()
    lag_one = values.autocorr(1) if len(values) >= 20 else np.nan

    # Daily structure does not require a one-second STL fit. Aggregate
    # high-frequency recordings to one-minute medians before decomposition.
    seasonal_cadence = max(float(cadence), 60.0)
    seasonal_values = values
    if seasonal_cadence > float(cadence):
        seasonal_values = values.resample(
            pd.Timedelta(seconds=seasonal_cadence)
        ).median().dropna()
    daily_period = round(86400 / seasonal_cadence)
    stl_status, seasonal_strength = "insufficient_span", np.nan
    if (daily_period >= 2 and len(seasonal_values) >= 6 * daily_period
            and seasonal_values.nunique() > 2):
        result = STL(seasonal_values, period=daily_period, robust=True).fit()
        denominator = np.var(result.seasonal + result.resid)
        seasonal_strength = max(0, 1 - np.var(result.resid) / denominator) if denominator else 0
        stl_status = "evaluated"
    temporal_rows.append({
        "metric_id": metric_id, "entity_id": chosen.entity_id,
        "regular_observations": len(values), "lag_1_acf": lag_one,
        "seasonality_cadence_seconds": seasonal_cadence,
        "daily_stl_status": stl_status, "daily_seasonal_strength": seasonal_strength,
    })
    if len(values) >= 100:
        fig, axis = plt.subplots(figsize=(8, 3))
        plot_acf(values.iloc[:10000], lags=min(60, len(values) // 4), zero=False, ax=axis)
        axis.set_title(f"{metric_id}: ACF on longest regular segment")
        plt.tight_layout(); plt.show()

temporal_summary = pd.DataFrame(temporal_rows)
display(temporal_summary)


## 5. Frozen-reference review and channel readiness

An unusual or unstable calibration centre is a review flag, not proof of a
faulty asset. Flagged entity-metric references fall back to the pooled
reference; the entity remains monitored. Readiness is reported per metric and
proposed detector channel so limited sensor availability cannot flatter the
later recall denominator.


In [ ]:
entity_medians = connection.execute(f"""
    SELECT CAST(entity_id AS VARCHAR) AS entity_id, metric_id,
           median(value) FILTER (WHERE quality_code <> 'invalid') AS entity_median
    FROM telemetry WHERE {where}
    GROUP BY entity_id, metric_id
""", parameters).df()
fleet = entity_medians.groupby("metric_id")["entity_median"].agg(
    fleet_median="median",
    q25=lambda x: x.quantile(0.25),
    q75=lambda x: x.quantile(0.75),
).reset_index()
baseline_review = entity_medians.merge(fleet, on="metric_id", how="left")
scale = (baseline_review.q75 - baseline_review.q25) / 1.349
baseline_review["fleet_robust_z"] = (
    (baseline_review.entity_median - baseline_review.fleet_median)
    / scale.where(scale.gt(0))
)
ordered = sample.sort_values(["entity_id", "metric_id", "event_ts"]).copy()
groups = ordered.groupby(["entity_id", "metric_id"])
ordered["position"] = groups.cumcount()
ordered["series_rows"] = groups["value"].transform("size")
ordered["subperiod"] = np.where(
    ordered.position < ordered.series_rows / 2, "early", "late"
)
stability = (
    ordered.groupby(["entity_id", "metric_id", "subperiod"])["value"]
    .agg(["median", "size"]).unstack("subperiod")
)
stability.columns = [f"{measure}_{period}" for measure, period in stability.columns]
stability = stability.reset_index()
for column in ["median_early", "median_late", "size_early", "size_late"]:
    if column not in stability:
        stability[column] = np.nan
baseline_review = baseline_review.merge(
    stability, on=["entity_id", "metric_id"], how="left"
)
enough_history = baseline_review[["size_early", "size_late"]].min(axis=1).ge(10)
baseline_review["stability_robust_z"] = (
    (baseline_review.median_late - baseline_review.median_early).abs()
    / scale.where(scale.gt(0))
).where(enough_history)
gross_outlier = baseline_review.fleet_robust_z.abs().gt(8)
unstable_centre = baseline_review.stability_robust_z.gt(4)
baseline_review["review_flag"] = gross_outlier | unstable_centre
baseline_review["review_reason"] = np.select(
    [gross_outlier & unstable_centre, gross_outlier, unstable_centre],
    ["fleet_outlier_and_unstable", "fleet_outlier", "unstable_centre"],
    default="none",
)

minimum_rows = {"rapid_residual": 20, "drift_cusum": 50, "dispersion_change": 50,
                "pca_spe": 50, "isolation_forest": 50}
readiness = pd.concat([
    series_summary.assign(
        channel=channel,
        status=np.where(
            series_summary.rows.lt(required), "warming_up",
            np.where(series_summary.valid_rate.lt(0.80), "temporarily_unscoreable", "monitored"),
        ),
    )
    for channel, required in minimum_rows.items()
], ignore_index=True)
readiness_summary = readiness.groupby(["metric_id", "channel", "status"], as_index=False).size()

display(baseline_review.loc[baseline_review.review_flag].head(30))
display(readiness_summary)


## 6. Save the modelling hand-off


In [ ]:
cadences = sorted(pd.to_numeric(catalogue.expected_cadence_seconds, errors="coerce").dropna().unique())
base_cadence = int(np.median(cadences))
settings = {
    "telecom": {"dispersion_window_seconds": 24 * 3600, "case_gap_seconds": 3600},
    "petrobras_3w": {"dispersion_window_seconds": 300, "case_gap_seconds": 0},
}[SECTOR]
decisions = {
    "eda_version": EDA_VERSION,
    "sector": SECTOR,
    "analysis_partition": "calibration",
    "canonical_fingerprint": manifest["fingerprint"],
    "primary_split": primary_split,
    "base_cadence_seconds": base_cadence,
    **settings,
    "reference_model": "calibration-frozen median and robust scale",
    "measurement_kind_transformations": {
        "gauge": "level and first difference",
        "bounded_fraction": "zero-safe log10 level and first difference",
        "interval_count": "log1p level and first difference",
        "cumulative_counter": "non-negative increment plus reset flag",
        "discrete_state": "state and transition indicator",
        "clipped": "withhold value from asset-health features; retain indicator",
    },
    "seasonality_enabled_in_v1": False,
    "baseline_review_flags": int(baseline_review.review_flag.sum()),
    "baseline_review_action": "flagged entity-metric references use pooled fallback",
    "baseline_stability_method": "early-versus-late median on reproducible calibration sample",
}

with new_output_directory(EDA_ROOT) as output:
    metric_summary.to_csv(output / "metric_summary.csv", index=False)
    temporal_summary.to_csv(output / "temporal_summary.csv", index=False)
    baseline_review.to_parquet(output / "baseline_review.parquet", index=False)
    readiness.to_parquet(output / "readiness.parquet", index=False)
    write_json(output / "eda_decisions.json", decisions)

display(pd.Series(decisions, name="decision").to_frame())
print("Saved:", EDA_ROOT)
print("Next: 03_EVALUATION_HARNESS.ipynb")
connection.close()
